In [4]:
from astroquery.alma import Alma
from astroquery import alma
from astropy.coordinates import SkyCoord
import astropy.units as u

query_results = Alma.query_region(SkyCoord.from_name('NGC 6334 C'), radius=0.2*u.deg)

etbl = alma.get_enhanced_table(query_results)

DALServiceError: 502 Server Error: Proxy Error for url: https://almascience.nrao.edu/tap/sync/z0xgx6lmeiv72qyo/run

In [ ]:
import pyavm
import PIL
img = PIL.Image.open('/orange/adamginsburg/jwst/outreach_pngs/NGC6334_JWST_colorcomposite.png')
avm = pyavm.AVM.from_image('/orange/adamginsburg/jwst/outreach_pngs/NGC6334_JWST_colorcomposite.png')

In [ ]:
colors = {3: 'y',
          6: 'g',
          7: 'orange',
          8: 'r',
          4: 'b',
          9: 'c',
          5: 'm',
          10: 'k',
          1: 'w',
         }

import pylab as pl
import numpy as np
ww = avm.to_wcs()#[::4,::4]
ax = pl.subplot(projection=ww)
#ax.imshow(np.array(img)[::4,::4,:])
ax.imshow(img)
for row in etbl:
    reg = row['s_region'].to_pixel(ww)
    band = int(row['band_list'])
    try:
        reg.plot(ax=ax, edgecolor=colors[band])
    except Exception as ex:
        print(ex)
#ax.axis([-10000, 20000, -10000, 20000])

In [ ]:
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import simple_norm
fh = fits.open('/orange/adamginsburg/localclouds/ngc6334/member.uid___A001_X2d20_X10f5.NGC6334_South_sci.spw25_27_29_31_33_35_37.cont.I.pbcor.fits')

ww_alma = WCS(fh[0].header).celestial
pl.figure(dpi=200);
ax = pl.subplot(projection=ww_alma)
im = ax.imshow(fh[0].data.squeeze(), norm=simple_norm(fh[0].data.squeeze(), vmin=0, stretch='asinh', max_percent=99.9));
pl.colorbar(mappable=im);
#ax.imshow(img, projection=ax.get_transform(ww))
ax.axis([0, 700, 1300, 2000]);

In [ ]:
pl.figure(dpi=200)
ww = avm.to_wcs()
ax = pl.subplot(projection=ww)
ax.imshow(img)

ww_alma = WCS(fh[0].header).celestial
ax.contour(fh[0].data.squeeze(), transform=ax.get_transform(ww_alma), linewidths=[1.5]*5, colors=['w']*5,
           levels=[0.0005, 0.005, 0.01, 0.03, 0.05],
           norm=simple_norm(fh[0].data.squeeze(), vmin=0, max_percent=99, stretch='asinh')
          );
ax.contour(fh[0].data.squeeze(), transform=ax.get_transform(ww_alma), linewidths=[0.5]*5, colors=['k']*5,
           levels=[0.0005, 0.005, 0.01, 0.03, 0.05],
           norm=simple_norm(fh[0].data.squeeze(), vmin=0, max_percent=99, stretch='asinh')
          );
ax.axis([-500,6000,-500,7000]);

In [ ]:
%%bash
rm /orange/adamginsburg/localclouds/ngc6334/*_bp*
rm /orange/adamginsburg/localclouds/ngc6334/*_ph*

In [ ]:
ls /orange/adamginsburg/localclouds/ngc6334/

In [ ]:
uid_url_table = Alma.get_data_info(etbl['member_ous_uid'], expand_tarfiles=True)

fits_urls = [url for url in uid_url_table['access_url'] if '.fits' in url and '_sci' in url and '.pb.' not in url
             and 'cube' not in url and 'cont.I.pbcor' in url]
print(fits_urls)

filelist = Alma.download_files(fits_urls, savedir='/orange/adamginsburg/localclouds/ngc6334/')

In [ ]:
if False:
    uid_url_table = Alma.get_data_info(etbl['member_ous_uid'][0], expand_tarfiles=True)
    
    fits_urls = [url for url in uid_url_table['access_url'] if '.fits' in url and '_sci' in url and '.pb.' not in url
                 and 'cube' in url]
    
    filelist = Alma.download_files(fits_urls, savedir='/orange/adamginsburg/localclouds/ngc6334/')